# Extracción y Análisis de Datos de Reddit sobre Memecoins  
**Objetivo**: Extraer posts de subreddits de criptomonedas (DOGE, SHIB, PEPE) y analizar su relación con la volatilidad.  
**Fuente**: API de Reddit via PRAW.

### 1. Instalación y carga de librerías

In [1]:
!pip install praw pandas matplotlib tqdm beautifulsoup4 requests
import praw
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import re
import time

### 2. Configuración API Reddit (PRAW)

In [5]:
reddit = praw.Reddit(
    client_id="zlLm6fqrukLXg-cDv5MwYQ",
    client_secret="bxL5btHBR-9zZDu5ybFdx-jrQVlqrw",
    user_agent="TFM_Memecoins/1.0 (by /Individual-Lab6313)"
)
print(f"Conexión exitosa. Usuario: {reddit.user.me()}")

Conexión exitosa. Usuario: None


### 3. Función para extraer posts de Reddit

In [6]:
def get_reddit_posts(subreddit_name, limit=10000):
    subreddit = reddit.subreddit(subreddit_name)
    posts = []
    
    for post in tqdm(subreddit.new(limit=limit), desc=f"Extrayendo posts de r/{subreddit_name}"):
        created = datetime.utcfromtimestamp(post.created_utc)
        posts.append({
            "title": post.title,
            "date": created.strftime('%Y-%m-%d %H:%M:%S'),
            "source": f"reddit/{subreddit_name}",
            "url": f"https://reddit.com{post.permalink}",
            "text": post.selftext if post.selftext else ""
        })
    print(f"Total posts obtenidos de r/{subreddit_name}: {len(posts)}")
    return posts

# Ejemplo para extraer posts de los subreddits deseados
subreddits = ["CryptoMarkets", "CryptoCurrency", "CryptoNews"]
all_reddit_posts = []
for sub in subreddits:
    all_reddit_posts.extend(get_reddit_posts(sub))
    
df_reddit = pd.DataFrame(all_reddit_posts)


Extrayendo posts de r/CryptoMarkets: 725it [00:11, 62.45it/s]


Total posts obtenidos de r/CryptoMarkets: 725


Extrayendo posts de r/CryptoCurrency: 755it [00:11, 66.16it/s]


Total posts obtenidos de r/CryptoCurrency: 755


Extrayendo posts de r/CryptoNews: 169it [00:02, 81.67it/s]

Total posts obtenidos de r/CryptoNews: 169


### 4. Función para scrapear noticias de globalissues.org

In [25]:
import requests
from bs4 import BeautifulSoup
import time
from tqdm import tqdm

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
}

def get_response_with_retries(url, max_retries=5):
    for i in range(max_retries):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            return response
        except Exception as e:
            print(f"Error al obtener {url}: {e}. Intento {i+1}/{max_retries}")
            time.sleep(2 ** i)
    return None

def scrape_globalissues_2024(start_page=1, end_page=240):
    all_news = []
    base_url = "https://www.globalissues.org/news/2024/page/"
    
    for page in tqdm(range(start_page, end_page + 1), desc="Scraping globalissues 2024"):
        url = f"{base_url}{page}"
        response = get_response_with_retries(url)
        if response is None:
            print(f"Fallo definitivo en página {page}, la salto.")
            continue
        
        soup = BeautifulSoup(response.text, "html.parser")
        news_blocks = soup.select("ol.intro-blocks > li.page-intro")
        
        for block in news_blocks:
            title_tag = block.find("h2")
            title = title_tag.get_text(strip=True) if title_tag else "No title found"
            
            date_tag = block.find("p", class_="page-intro-summary-info")
            time_tag = date_tag.find("time") if date_tag else None
            if time_tag and time_tag.has_attr('datetime'):
                date = time_tag['datetime']
            elif time_tag:
                date = time_tag.get_text(strip=True)
            else:
                date = None
            
            all_news.append({
                "title": title,
                "date": date,
                "source": "globalissues_2024",
                "url": url,
                "text": ""
            })
        time.sleep(1)  # Aumenta la espera
    return all_news

# Ejecutar extracción
news_2024 = scrape_globalissues_2024(1, 240)
df_globalissues_2024 = pd.DataFrame(news_2024)


Scraping globalissues 2024:  20%|██        | 48/240 [01:13<04:34,  1.43s/it]

Error al obtener https://www.globalissues.org/news/2024/page/49: HTTPSConnectionPool(host='www.globalissues.org', port=443): Read timed out. (read timeout=10). Intento 1/5


Scraping globalissues 2024:  25%|██▌       | 60/240 [01:42<04:29,  1.50s/it]

Error al obtener https://www.globalissues.org/news/2024/page/61: HTTPSConnectionPool(host='www.globalissues.org', port=443): Read timed out. (read timeout=10). Intento 1/5


Scraping globalissues 2024:  40%|████      | 96/240 [02:47<03:32,  1.47s/it]

...

Scraping globalissues 2024:  83%|████████▎ | 199/240 [06:40<01:15,  1.84s/it]

Error al obtener https://www.globalissues.org/news/2024/page/200: HTTPSConnectionPool(host='www.globalissues.org', port=443): Read timed out. (read timeout=10). Intento 1/5


Scraping globalissues 2024:  96%|█████████▋| 231/240 [07:40<00:12,  1.44s/it]

Error al obtener https://www.globalissues.org/news/2024/page/232: HTTPSConnectionPool(host='www.globalissues.org', port=443): Read timed out. (read timeout=10). Intento 1/5


Scraping globalissues 2024: 100%|██████████| 240/240 [08:04<00:00,  2.02s/it]


### 5. Función para scrapear noticias 2025

In [32]:
import pandas as pd

def scrape_globalissues_2025(start_page=1, end_page=119):
    all_news = []
    base_url = "https://www.globalissues.org/news/page/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }

    for page in tqdm(range(start_page, end_page + 1), desc="Scraping globalissues 2025"):
        url = f"{base_url}{page}"
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code != 200:
                print(f"Error al obtener página {page}, status {response.status_code}")
                continue
        except Exception as e:
            print(f"Error al conectar con {url}: {e}")
            continue
        
        soup = BeautifulSoup(response.text, "html.parser")
        news_blocks = soup.select("ol.intro-blocks > li.page-intro")

        print(f"\nPágina {page} - Encontradas {len(news_blocks)} noticias:")
        for block in news_blocks:
            # TITULAR
            title_tag = block.find("h2")
            title = title_tag.get_text(strip=True) if title_tag else "No title found"

            # FECHA original
            date_tag = block.find("p", class_="page-intro-summary-info")
            time_tag = date_tag.find("time") if date_tag else None
            if time_tag and time_tag.has_attr('datetime'):
                raw_date = time_tag['datetime']
            elif time_tag:
                raw_date = time_tag.get_text(strip=True)
            else:
                raw_date = None

            # Normalizar fecha a formato ISO o None
            if raw_date is not None:
                # Intentamos parsear con pandas
                date_parsed = pd.to_datetime(raw_date, errors='coerce', utc=True)
                if pd.isna(date_parsed):
                    date = None
                else:
                    # Guardamos como string ISO 8601 sin zona horaria (opcional)
                    date = date_parsed.isoformat()
            else:
                date = None

            print(f"TITULAR: {title}")
            print(f"FECHA: {date}\n")

            all_news.append({
                "title": title,
                "date": date,
                "source": "globalissues_2025",
                "url": url,
                "text": ""
            })

        time.sleep(1)  # Espera para no saturar el servidor
    return all_news

# Ejecutar extracción para 2025 con impresión
news_2025 = scrape_globalissues_2025(1, 119)
df_globalissues_2025 = pd.DataFrame(news_2025)


Scraping globalissues 2025:   0%|          | 0/119 [00:00<?, ?it/s]


Página 1 - Encontradas 10 noticias:
TITULAR: Economic Growth is the Wrong Metric for Our Time
FECHA: 2025-05-23T14:58:59+00:00

TITULAR: Human Life Hinges on the Preservation of Biological Diversity
FECHA: 2025-05-23T14:16:07+00:00

TITULAR: In Harmony with Nature: A Dryland Perspective on Development and Biodiversity
FECHA: 2025-05-23T13:08:12+00:00

TITULAR: The Largest Multi-Billion Dollar Deal in US History – & a Potential 51st American State?
FECHA: 2025-05-23T12:10:53+00:00

TITULAR: ‘A silent crisis’: Obstetric fibrosis affects 500,000 women, yet it’s fully treatable
FECHA: 2025-05-23T12:00:00+00:00

TITULAR: Syrians face staggering needs amid insecurity and healthcare crisis
FECHA: 2025-05-23T12:00:00+00:00

TITULAR: Aid teams highlight growing anxiety in Gaza after food is looted
FECHA: 2025-05-23T12:00:00+00:00

TITULAR: How Computational Biology Is Zoning in on the Future of Agriculture
FECHA: 2025-05-23T01:54:43+00:00

TITULAR: Global Push to Protect Oceans Gains Momentum 

Scraping globalissues 2025:   1%|          | 1/119 [00:02<04:22,  2.23s/it]


Página 2 - Encontradas 10 noticias:
TITULAR: Using AI as an Ally: What the latest UNDP Human Development Report Means for Latin America, Caribbean
FECHA: 2025-05-22T18:44:11+00:00

TITULAR: HeForShe Campaign Tackles 'Sex for Fish' Abuse Malawis Lakeshore Communities
FECHA: 2025-05-22T18:16:05+00:00

TITULAR: A New Pope at a Pivotal Moment: Civil Society’s Hopes for Leo XIV
FECHA: 2025-05-22T14:49:36+00:00

TITULAR: The True Cost of America's Retreat: How USAID Cuts Threaten Millions of Lives
FECHA: 2025-05-22T12:53:53+00:00

TITULAR: UN Reform – Once Again?
FECHA: 2025-05-22T12:25:34+00:00

TITULAR: World News in Brief: UK cedes sovereignty over Chagos Islands, suffering in Sudan deepens, UN releases new emergency relief funds
FECHA: 2025-05-22T12:00:00+00:00

TITULAR: Civilians ‘trapped and terrorised’: UN sounds alarm over collapse of critical protections
FECHA: 2025-05-22T12:00:00+00:00

TITULAR: Gaza: Strikes on houses and tents responsible for over half of deaths this week
FECHA:

Scraping globalissues 2025:   2%|▏         | 2/119 [00:03<03:24,  1.75s/it]

...


Página 117 - Encontradas 10 noticias:
TITULAR: Ukraine: Zaporizhzhia attack marks highest civilian casualties in nearly two years
FECHA: 2025-01-09T12:00:00+00:00

TITULAR: UN chief offers condolences amid devastating wildfires in California
FECHA: 2025-01-09T12:00:00+00:00

TITULAR: Global growth to remain subdued in 2025 amid uncertainty, UN report warns
FECHA: 2025-01-09T12:00:00+00:00

TITULAR: Lebanon: Election of new president a long-awaited first step, senior UN official says
FECHA: 2025-01-09T12:00:00+00:00

TITULAR: More than 125,000 refugees return to Syria in desperate conditions
FECHA: 2025-01-09T12:00:00+00:00

TITULAR: Colombias Historic Child Marriage Ban
FECHA: 2025-01-09T03:25:47+00:00

TITULAR: Erratic Sales and Government Apathy Hurt Telangana Weavers
FECHA: 2025-01-08T15:46:48+00:00

TITULAR: Sudan's Humanitarian Crisis Expected to Worsen in 2025
FECHA: 2025-01-08T13:47:59+00:00

TITULAR: Our Health is at Stake: The Solutions SIDS Need to Fight Climate Change
FECHA

Scraping globalissues 2025:  98%|█████████▊| 117/119 [03:33<00:02,  1.44s/it]


Página 118 - Encontradas 10 noticias:
TITULAR: Rights experts call for immediate release of Abu Zubaydah from Guantánamo
FECHA: 2025-01-08T12:00:00+00:00

TITULAR: DPR Korea’s nuclear quest thwarts disarmament efforts, Security Council hears
FECHA: 2025-01-08T12:00:00+00:00

TITULAR: Ukraine in grip of third winter of escalating Russian attacks
FECHA: 2025-01-08T12:00:00+00:00

TITULAR: MIDDLE EAST LIVE: Security Council meets on Syria, plus Gaza and Lebanon updates
FECHA: 2025-01-08T12:00:00+00:00

TITULAR: Genocidal President, Genocidal Politics
FECHA: 2025-01-07T15:07:27+00:00

TITULAR: Is Bangladesh's Currency Reprint Pressing Delete on Bangabandhu's Legacy?
FECHA: 2025-01-07T14:47:55+00:00

TITULAR: Current Financing for Development Priorities Today
FECHA: 2025-01-07T14:11:56+00:00

TITULAR: Gaza: Humanitarians assist families impacted by recent airstrike in Deir Al-Balah
FECHA: 2025-01-07T12:00:00+00:00

TITULAR: World News in Brief: Killings of Alawites in Syria, executions in 

Scraping globalissues 2025: 100%|██████████| 119/119 [03:44<00:00,  1.89s/it]

Error al conectar con https://www.globalissues.org/news/page/119: HTTPSConnectionPool(host='www.globalissues.org', port=443): Read timed out. (read timeout=10)


### 6. Unificar todos los datasets y limpieza básica

In [33]:
df_news = pd.concat([df_reddit, df_globalissues_2024, df_globalissues_2025], ignore_index=True)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\[.*?\]\(.*?\)", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

df_news["title"] = df_news["title"].apply(clean_text)
df_news["text"] = df_news["text"].apply(clean_text)

# Convertir fecha con manejo UTC
df_news["date"] = pd.to_datetime(df_news["date"], errors="coerce", utc=True)

print("Fechas inválidas (NaT):", df_news["date"].isna().sum())

print("Filas antes de eliminar duplicados:", len(df_news))
df_news.drop_duplicates(subset=["title", "date"], inplace=True)
print("Filas después de eliminar duplicados:", len(df_news))

df_news.to_csv("news.csv", index=False, encoding="utf-8")
print("Dataset final guardado en 'news.csv'")


# Eliminar duplicados
df_news.drop_duplicates(subset=["title", "date"], inplace=True)

print(f"Total noticias tras limpieza: {len(df_news)}")

df_news.head()


Fechas inválidas (NaT): 3530
Filas antes de eliminar duplicados: 5179
Filas después de eliminar duplicados: 5174
Dataset final guardado en 'news.csv'
Total noticias tras limpieza: 5174


,title,date,source,url,text
0,freedom with strings attached,2025-05-23 13:12:50+00:00,reddit/CryptoMarkets,https://reddit.com/r/CryptoMarkets/comments/1k...,crypto projects promise freedom but then lock ...
1,why isnt mstr stock increasing in value,2025-05-23 12:18:42+00:00,reddit/CryptoMarkets,https://reddit.com/r/CryptoMarkets/comments/1k...,bitcoin has been surging but mstr has barely m...
2,260 million cetus hack on sui triggers network...,2025-05-23 11:33:10+00:00,reddit/CryptoMarkets,https://reddit.com/r/CryptoMarkets/comments/1k...,
3,how do pro crypto traders identify and trade p...,2025-05-23 11:17:16+00:00,reddit/CryptoMarkets,https://reddit.com/r/CryptoMarkets/comments/1k...,from what ive observed many professional crypt...
4,daily discussion megathread may 23 2025 gmt0,2025-05-23 11:00:36+00:00,reddit/CryptoMarkets,https://reddit.com/r/CryptoMarkets/comments/1k...,this post contains content not supported on ol...


### 7. Guardar dataset final

In [27]:
df_news.to_csv("news.csv", index=False, encoding="utf-8")
print("Dataset final guardado en 'news.csv'")


Dataset final guardado en 'news.csv'
